# Ad Click Prediction
A predictive system that determines the likelihood of a user clicking on an advertisement based on:

Phase 1: Data Exploration & Understanding
--

Step 1: Initial Data Loading

In [31]:
# Read the messy training data CSV file and display the first fe_dfw rows of the DataFrame.
from pathlib import Path
import pandas as pd

csv_path = Path('/Data/Ad_click_prediction_train.csv')
if not csv_path.exists():
    csv_path = Path('Data/Ad_click_prediction_train.csv')

df = pd.read_csv(csv_path)

print("Sales Campaign Dataset Overview:")
print("=" * 80)
display(df.head())

print("Dataset dimensions:", df.shape)
print("Column types:\n", df.dtypes)
print("Target variable (CTR) distribution:\n", df['is_click'].value_counts())
print("Missing values:\n", df.isnull().sum())

Sales Campaign Dataset Overview:


,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,age_level,user_depth,city_development_index,var_1,is_click
0,140690,2017-07-02 00:00,858557,C,359520,13787,4,NaN,10.0,Female,4.0,3.0,3.0,0,0
1,333291,2017-07-02 00:00,243253,C,105960,11085,5,NaN,8.0,Female,2.0,2.0,NaN,0,0
2,129781,2017-07-02 00:00,243253,C,359520,13787,4,NaN,8.0,Female,2.0,2.0,NaN,0,0
3,464848,2017-07-02 00:00,1097446,I,359520,13787,3,NaN,3.0,Male,3.0,3.0,2.0,1,0
4,90569,2017-07-02 00:01,663656,C,405490,60305,3,NaN,2.0,Male,2.0,3.0,2.0,1,0


Dataset dimensions: (463291, 15)
Column types:
 session_id                  int64
DateTime                      str
user_id                     int64
product                       str
campaign_id                 int64
webpage_id                  int64
product_category_1          int64
product_category_2        float64
user_group_id             float64
gender                        str
age_level                 float64
user_depth                float64
city_development_index    float64
var_1                       int64
is_click                    int64
dtype: object
Target variable (CTR) distribution:
 is_click
0    431960
1     31331
Name: count, dtype: int64
Missing values:
 session_id                     0
DateTime                       0
user_id                        0
product                        0
campaign_id                    0
webpage_id                     0
product_category_1             0
product_category_2        365854
user_group_id              18243
gender            

Step 2: Exploratory Data Analysis (EDA)

In [32]:
# Target Distribution

## What percentage of ads get clicked?
display(df['is_click'].value_counts(normalize=True))

## Is the dataset severely imbalanced?
clicked_percentage = round(df['is_click'].mean() * 100, 2)
print("Percentage of ads clicked:", clicked_percentage, "%")
print("Percentage of ads not clicked:", round(100 - clicked_percentage, 2), "%")
print("Is the dataset severely imbalanced?", "Yes" if clicked_percentage < 5 else "No")

## Do you need resampling techniques?

is_click
0    0.932373
1    0.067627
Name: proportion, dtype: float64

Percentage of ads clicked: 6.76 %
Percentage of ads not clicked: 93.24 %
Is the dataset severely imbalanced? No


In [33]:
#Temporal Patterns

## Create a new column for the hour of the day from the timestamp
df['hour'] = pd.to_datetime(df['DateTime']).dt.hour
df['month'] = pd.to_datetime(df['DateTime']).dt.month
df['is_weekend'] = (pd.to_datetime(df['DateTime']).dt.dayofweek >= 5).astype(int)
display(df[['DateTime', 'hour', 'month', 'is_weekend']].head())

print("Which hours have highest click rates?")
display(df.groupby('hour')['is_click'].mean().sort_values(ascending=False).head(5))
print("Late night and early morning hours have the highest click rates.")

print("Are weekends different from weekdays?")
display(df.groupby('is_weekend')['is_click'].mean())
print("Click rates are higher on weekends compared to weekdays.")

print("Do certain months perform better?")
print('All months ', df['month'].unique())
display(df.groupby('month')['is_click'].mean().sort_values(ascending=False).head(5))
print("Data is available only for one month.")

,DateTime,hour,month,is_weekend
0,2017-07-02 00:00,0,7,1
1,2017-07-02 00:00,0,7,1
2,2017-07-02 00:00,0,7,1
3,2017-07-02 00:00,0,7,1
4,2017-07-02 00:01,0,7,1


Which hours have highest click rates?


hour
1    0.074608
7    0.073978
6    0.072822
8    0.070271
9    0.070101
Name: is_click, dtype: float64

Late night and early morning hours have the highest click rates.
Are weekends different from weekdays?


is_weekend
0    0.066468
1    0.073262
Name: is_click, dtype: float64

Click rates are higher on weekends compared to weekdays.
Do certain months perform better?
All months  [7]


month
7    0.067627
Name: is_click, dtype: float64

Data is available only for one month.


In [34]:
## User Behavior

### Do certain age groups click more?
print("Which age groups have highest click rates?")
print('=' * 80)
print('All age levels ', df['age_level'].unique())
display(df.groupby('age_level')['is_click'].mean().sort_values(ascending=False).head(5))
print("Youngest & oldest age groups tend to click more on ads.")

### Is there a gender difference in click rates?
print("Click rates by gender:")
print('=' * 80)
display(df.groupby('gender')['is_click'].mean())
print("Click rates are higher for males compared to females.")

### How does user group affect clicking?
print("Which user groups have highest click rates?")
print('=' * 80)
print('All user groups ', df['user_group_id'].unique())
display(df.groupby('user_group_id')['is_click'].mean().sort_values(ascending=False).head(5))
print("Highest & lowest user groups have significantly higher click rates than others.")

Which age groups have highest click rates?
All age levels  [ 4.  2.  3.  1. nan  5.  6.  0.]


age_level
0.0    0.084967
6.0    0.082276
1.0    0.074803
5.0    0.074153
2.0    0.070919
Name: is_click, dtype: float64

Youngest & oldest age groups tend to click more on ads.
Click rates by gender:


gender
Female    0.064445
Male      0.067942
Name: is_click, dtype: float64

Click rates are higher for males compared to females.
Which user groups have highest click rates?
All user groups  [10.  8.  3.  2.  1.  9.  4. nan 11.  7.  5. 12.  6.  0.]


user_group_id
12.0    0.088889
0.0     0.084967
6.0     0.078306
11.0    0.076706
1.0     0.075144
Name: is_click, dtype: float64

Highest & lowest user groups have significantly higher click rates than others.


In [35]:
## Campaign Performance

### Which campaigns have highest CTR?
print("Which campaigns have highest click rates?")
print('Total campaigns ', df['campaign_id'].nunique())
print('Top 5 campaigns:')
print('=' * 80)
display(df.groupby('campaign_id')['is_click'].mean().sort_values(ascending=False).head(5))

### Which products get more clicks?
print("Top 5 products with highest click rates:")
print('=' * 80)
display(df.groupby('product')['is_click'].mean().sort_values(ascending=False).head(5))

### Do certain webpages convert better?
print("Top 5 webpages with highest click rates:")
print('=' * 80)
display(df.groupby('webpage_id')['is_click'].mean().sort_values(ascending=False).head(5))

Which campaigns have highest click rates?
Total campaigns  10
Top 5 campaigns:


campaign_id
405490    0.091307
404347    0.077534
98970     0.076829
396664    0.072624
105960    0.068345
Name: is_click, dtype: float64

Top 5 products with highest click rates:


product
J    0.092700
D    0.071815
H    0.069852
C    0.069149
E    0.068712
Name: is_click, dtype: float64

Top 5 webpages with highest click rates:


webpage_id
60305    0.091307
53587    0.077534
6970     0.076829
51181    0.072624
11085    0.068345
Name: is_click, dtype: float64

Phase 2: Feature Engineering
---

In [36]:
## DateTime Feature Extraction
print("Extracting date and time features from the dataset.")

# - hour: Hour of day (0-23)
# - day_of_week: Day (0=Monday, 6=Sunday)
df['day_of_week'] = pd.to_datetime(df['DateTime']).dt.dayofweek
# - day_of_month: Date of month (1-31)
df['day_of_month'] = pd.to_datetime(df['DateTime']).dt.day
# time_of_day: Categorical (night/morning/afternoon/evening)
df['time_of_day'] = pd.cut(df['hour'], bins=[-1, 5, 11, 17, 23], labels=['night', 'morning', 'afternoon', 'evening'])

display(df.head())

Extracting date and time features from the dataset.


,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,...,user_depth,city_development_index,var_1,is_click,hour,month,is_weekend,day_of_week,day_of_month,time_of_day
0,140690,2017-07-02 00:00,858557,C,359520,13787,4,NaN,10.0,Female,...,3.0,3.0,0,0,0,7,1,6,2,night
1,333291,2017-07-02 00:00,243253,C,105960,11085,5,NaN,8.0,Female,...,2.0,NaN,0,0,0,7,1,6,2,night
2,129781,2017-07-02 00:00,243253,C,359520,13787,4,NaN,8.0,Female,...,2.0,NaN,0,0,0,7,1,6,2,night
3,464848,2017-07-02 00:00,1097446,I,359520,13787,3,NaN,3.0,Male,...,3.0,2.0,1,0,0,7,1,6,2,night
4,90569,2017-07-02 00:01,663656,C,405490,60305,3,NaN,2.0,Male,...,3.0,2.0,1,0,0,7,1,6,2,night


In [37]:
# 2. Interaction Features
# Why: Combinations of features often reveal hidden patterns.

def create_interaction_features(df, features):
    for feature_pair in features:
        new_feature_name = '_'.join(feature_pair)
        df[new_feature_name] = df[feature_pair[0]].astype(str) + '_' + df[feature_pair[1]].astype(str)
        display(df.groupby(new_feature_name)['is_click'].mean().sort_values(ascending=False).head(5))


# Features to Create:
## 1. user_product_interaction: user_id + product
## 2. campaign_webpage: campaign_id + webpage_id
## 3. gender_age: gender + age_level
create_interaction_features(df, [('user_id', 'product'), ('campaign_id', 'webpage_id'), ('gender', 'age_level')])


user_id_product
1118426_C    1.0
1118358_E    1.0
1118626_E    1.0
1118626_D    1.0
1118694_D    1.0
Name: is_click, dtype: float64

campaign_id_webpage_id
405490_60305    0.091307
404347_53587    0.077534
98970_6970      0.076829
396664_51181    0.072624
105960_11085    0.068345
Name: is_click, dtype: float64

gender_age_level
Male_0.0      0.100000
Female_6.0    0.088889
Male_6.0      0.078306
Female_5.0    0.076706
Male_1.0      0.075144
Name: is_click, dtype: float64

In [38]:
## 3. Aggregated Features
## Why: Historical performance is a strong predictor.

### User-Level Aggregations:

print("User-Level Aggregations:")
print("=" * 80)
# - user_total_views: How many ads has this user seen?
user_total_views = df.groupby('user_id')['is_click'].count().rename('user_total_views')
print("User total views:\n", user_total_views.sort_values(ascending=False).head())

# - user_total_clicks: How many times has this user clicked?
user_total_clicks = df.groupby('user_id')['is_click'].sum().rename('user_total_clicks')
print("\n\nUser total clicks:\n", user_total_clicks.sort_values(ascending=False).head())

# - user_ctr: User's personal click-through rate
user_ctr = (user_total_clicks / user_total_views).rename('user_ctr')
print("\n\nUser CTR:\n", user_ctr.sort_values(ascending=False).head())

# - user_sessions: Number of unique sessions per user
user_sessions = df.groupby('user_id')['session_id'].nunique().rename('user_sessions')
print("\n\nUser sessions:\n", user_sessions.sort_values(ascending=False).head(10))


User-Level Aggregations:
User total views:
 user_id
658554    255
297960    225
983136    187
422201    157
929999    143
Name: user_total_views, dtype: int64


User total clicks:
 user_id
252158    15
357044    13
222506    12
40010     12
691846    11
Name: user_total_clicks, dtype: int64


User CTR:
 user_id
246193    1.0
498394    1.0
67690     1.0
624593    1.0
582632    1.0
Name: user_ctr, dtype: float64


User sessions:
 user_id
658554     255
297960     225
983136     187
422201     157
929999     143
18434      141
378888     138
1045409    136
577452     135
678048     130
Name: user_sessions, dtype: int64


In [39]:
# Product-Level Aggregations
print("\n\nProduct-Level Aggregations:")
print("=" * 80)

## - product_views: Total times this product was shown
## - product_ctr: This product's historical click rate
product_views = df.groupby('product')['is_click'].count().rename('product_views')
print("Product views:\n", product_views.sort_values(ascending=False).head(5))

product_clicks = df.groupby('product')['is_click'].sum().rename('product_clicks')
print("\n\nProduct clicks:\n", product_clicks.sort_values(ascending=False).head(5))

product_ctr = ((product_clicks / product_views) * 100).round(2)
product_ctr = product_ctr.rename('product_ctr')
print("\n\nProduct CTR (%):\n", product_ctr.sort_values(ascending=False).head(5))



Product-Level Aggregations:
Product views:
 product
C    163501
H    109574
I     63711
D     41064
B     22479
Name: product_views, dtype: int64


Product clicks:
 product
C    11306
H     7654
I     4079
D     2949
E     1474
Name: product_clicks, dtype: int64


Product CTR (%):
 product
J    9.27
D    7.18
H    6.99
C    6.91
E    6.87
Name: product_ctr, dtype: float64


In [40]:
# **Campaign-Level Aggregations:**

print("\n\nCampaign-Level Aggregations:")
print("=" * 80)

## - campaign_views: Total impressions for this campaign
## - campaign_ctr: Campaign's historical performance
campaign_views = df.groupby('campaign_id')['is_click'].count().rename('campaign_views')
print("Campaign views:\n", campaign_views.sort_values(ascending=False).head(5))

campaign_clicks = df.groupby('campaign_id')['is_click'].sum().rename('campaign_clicks')
print("\n\nCampaign clicks:\n", campaign_clicks.sort_values(ascending=False).head(5))

campaign_ctr = ((campaign_clicks / campaign_views) * 100).round(2).rename('campaign_ctr')
print("\n\nCampaign CTR (%):\n", campaign_ctr.sort_values(ascending=False).head(5))



Campaign-Level Aggregations:
Campaign views:
 campaign_id
359520    108155
405490     95973
360936     51888
118601     35531
98970      35065
Name: campaign_views, dtype: int64


Campaign clicks:
 campaign_id
405490    8763
359520    6340
98970     2694
360936    2346
404347    2235
Name: campaign_clicks, dtype: int64


Campaign CTR (%):
 campaign_id
405490    9.13
404347    7.75
98970     7.68
396664    7.26
105960    6.83
Name: campaign_ctr, dtype: float64


Phase 3: Data Preprocessing
--

In [41]:
# Step 1: Handle Missing Values

## Numerical columns: Fill with median (robust to outliers)
## Categorical columns: Fill with mode (most frequent value)

In [42]:
# Fill user_group_id, age_level, user_depth, and	city_development_index with median values

import numpy as np

median_cols = ['user_group_id', 'age_level', 'user_depth', 'city_development_index']

for col in median_cols:
    median_value = df[col].median()
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(median_value)
    print(f"{col:<24s} filled {n_missing:>7,} nulls with median {median_value}")

print("=" * 80)
print("Remaining nulls in these columns:")
print(df[median_cols].isnull().sum().to_string())


# Fill gender randomly with "Male" / "Female".
# Sampled in proportion to the observed split (88% Male / 12% Female) so the
# imputation preserves the real distribution. For a flat coin flip instead,
# swap the p=... argument for p=[0.5, 0.5].
print("\n" + "=" * 80)
print("gender BEFORE:")
print(df['gender'].value_counts(dropna=False).to_string())

rng = np.random.default_rng(42)          # seeded -> reproducible across re-runs
gender_probs = df['gender'].value_counts(normalize=True)
missing_gender = df['gender'].isna()

df.loc[missing_gender, 'gender'] = rng.choice(
    gender_probs.index.to_numpy(),
    size=missing_gender.sum(),
    p=gender_probs.to_numpy(),
)

print(f"\nFilled {missing_gender.sum():,} nulls randomly "
      f"({gender_probs['Male']:.1%} Male / {gender_probs['Female']:.1%} Female)")
print("\ngender AFTER:")
print(df['gender'].value_counts(dropna=False).to_string())


user_group_id            filled  18,243 nulls with median 3.0
age_level                filled  18,243 nulls with median 3.0
user_depth               filled  18,243 nulls with median 3.0
city_development_index   filled 125,129 nulls with median 2.0
Remaining nulls in these columns:
user_group_id             0
age_level                 0
user_depth                0
city_development_index    0

gender BEFORE:
gender
Male      393454
Female     51594
NaN        18243

Filled 18,243 nulls randomly (88.4% Male / 11.6% Female)

gender AFTER:
gender
Male      409588
Female     53703


In [43]:
# Step 2: Encode Categorical Variables
## Approach: Label Encoding

from sklearn.preprocessing import LabelEncoder

# product_category_2 is an ID code (29 distinct values, 79% missing), not a quantity.
# LabelEncoder cannot handle NaN, so mark "missing" as its own category first.
df['product_category_2'] = df['product_category_2'].fillna(-1)

## Columns to Encode:
encode_cols = [
    'product',
    'campaign_id',
    'webpage_id',
    'product_category_1',
    'product_category_2',
    'gender',
    'user_group_id',
    'var_1',
    # All interaction features created in Phase 2
    'user_id_product',
    'campaign_id_webpage_id',
    'gender_age_level',
]

# Keep the fitted encoders: the test set must use the SAME mapping, so we
# reuse these later rather than fitting a fresh encoder on the test data.
encoders = {}

print(f"{'column':<26}{'unique':>9}   {'dtype before':<14} encoded range")
print("=" * 80)

for col in encode_cols:
    dtype_before = str(df[col].dtype)
    le = LabelEncoder()
    # astype(str) gives every column a consistent type to sort/encode on
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

    n = len(le.classes_)
    flag = "  <-- very high cardinality" if n > 1000 else ""
    print(f"{col:<26}{n:>9,}   {dtype_before:<14} 0..{n - 1:,}{flag}")

print("=" * 80)
print("Encoded", len(encode_cols), "columns. Encoders stored in `encoders` for reuse on the test set.")
display(df[encode_cols].head())


column                       unique   dtype before   encoded range
product                          10   str            0..9
campaign_id                      10   int64          0..9
webpage_id                        9   int64          0..8
product_category_1                5   int64          0..4
product_category_2               30   float64        0..29
gender                            2   str            0..1
user_group_id                    13   float64        0..12
var_1                             2   int64          0..1
user_id_product             245,846   str            0..245,845  <-- very high cardinality
campaign_id_webpage_id           10   str            0..9
gender_age_level                 15   str            0..14
Encoded 11 columns. Encoders stored in `encoders` for reuse on the test set.


,product,campaign_id,webpage_id,product_category_1,product_category_2,gender,user_group_id,var_1,user_id_product,campaign_id_webpage_id,gender_age_level
0,2,2,1,3,0,0,2,0,215527,2,4
1,2,0,0,4,0,0,11,0,66859,0,2
2,2,2,1,3,0,0,11,0,66859,2,2
3,8,2,1,2,0,1,6,1,22716,2,10
4,2,6,7,2,0,1,5,1,172537,6,9


In [ ]:
# Step 3: Feature Selection
# Columns to Drop:

drop_cols = ['DateTime', 'session_id', 'user_id']
df = df.drop(columns=drop_cols, errors='ignore')

print("Dropped:", drop_cols)
display("Remaining columns:", df.head(5))

Dropped: ['DateTime', 'session_id', 'user_id']


'Remaining columns:'

,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,age_level,user_depth,city_development_index,...,is_click,hour,month,is_weekend,day_of_week,day_of_month,time_of_day,user_id_product,campaign_id_webpage_id,gender_age_level
0,2,2,1,3,0,2,0,4.0,3.0,3.0,...,0,0,7,1,6,2,night,215527,2,4
1,2,0,0,4,0,11,0,2.0,2.0,2.0,...,0,0,7,1,6,2,night,66859,0,2
2,2,2,1,3,0,11,0,2.0,2.0,2.0,...,0,0,7,1,6,2,night,66859,2,2
3,8,2,1,2,0,6,1,3.0,3.0,2.0,...,0,0,7,1,6,2,night,22716,2,10
4,2,6,7,2,0,5,1,2.0,3.0,2.0,...,0,0,7,1,6,2,night,172537,6,9


In [ ]:
# Step 4: Train-Test Split
# Why Stratify? In imbalanced datasets, stratification ensures both train and test have similar CTR.

from sklearn.model_selection import train_test_split

# time_of_day is still a pandas `category` dtype (night/morning/afternoon/evening).
# sklearn needs numeric input, so encode it the same way as the other categoricals.
if str(df['time_of_day'].dtype) == 'category':
    le_tod = LabelEncoder()
    df['time_of_day'] = le_tod.fit_transform(df['time_of_day'].astype(str))
    encoders['time_of_day'] = le_tod
    print("Encoded time_of_day ->", dict(zip(le_tod.classes_, range(len(le_tod.classes_)))))

# Separate features from the target
X = df.drop(columns=['is_click'])
y = df['is_click']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,          # keeps the same click rate in both splits
    random_state=42,     # reproducible across re-runs
)

print("=" * 80)
print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

# Confirm stratification actually preserved the CTR
print("\nClick-through rate by split:")
print(f"  full dataset : {y.mean():.4%}  ({y.sum():,} clicks)")
print(f"  train        : {y_train.mean():.4%}  ({y_train.sum():,} clicks)")
print(f"  test         : {y_test.mean():.4%}  ({y_test.sum():,} clicks)")

print(f"\nFeatures used ({X.shape[1]}):")
print(list(X.columns))
